In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats

Which products are more sale before Christmas?
Hypotheses:
1. Before Christmas, the total_price(Quantity*UnitPrice) increases because pepole buy presents and celebrate

In [ ]:
df = pd.read_csv('../data/data.csv', encoding='cp1252')
df.head()

In [ ]:
df.shape

In [ ]:
df['month'] = pd.to_datetime(df['InvoiceDate']).dt.month
df['total_price'] = df['Quantity'] * df['UnitPrice']

In [ ]:
x = df.groupby('month')['total_price'].sum().reset_index()
sns.scatterplot(x='month', y='total_price', data=x)

In [ ]:
x_before_christmas = x.loc[x['month'].isin([11,12])]['total_price']
x_christmas_is_far_away = x.loc[~x['month'].isin([11,12])]['total_price']
t_stat, p_value = stats.ttest_ind(x_before_christmas, x_christmas_is_far_away, equal_var=True)   #

print(f"t-statystyka: {t_stat}")
print(f"p-value:      {p_value}")
if p_value < 0.05:
    print("→ There is a statistically significant difference between the means")
else:
    print("→ No significant difference between means")

In [ ]:
x2 = df.copy()
x2 = df.groupby(['Description', 'month'])['total_price'].sum().reset_index().sort_values('total_price', ascending=False)
x2['before_christmas'] = x2['month'].isin([11,12])
x2.head(10)

In [ ]:
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

x2 = df.copy()
monthly_sales = x2.groupby(['Description', 'month'])['total_price'].sum().reset_index()
monthly_sales['before_christmas'] = monthly_sales['month'].isin([11, 12])

results = []

for description, data in monthly_sales.groupby('Description'):
    sales_before = data[data['before_christmas'] == True]['total_price']
    sales_after  = data[data['before_christmas'] == False]['total_price']

    if len(sales_before) < 2 or len(sales_after) < 2:
        continue   # too few observations for t-test

    t_stat, p_val = stats.ttest_ind(sales_before, sales_after, equal_var=False)  # Welch

    results.append({
        'Description': description,
        'mean_before': sales_before.mean(),
        'mean_after':  sales_after.mean(),
        'diff': sales_before.mean() - sales_after.mean(),
        'total_before': sales_before.sum(),
        'total_after':  sales_after.sum(),
        'months_before': len(sales_before),
        'months_after':  len(sales_after),
        't_stat': t_stat,
        'p_value': p_val
    })

df_results = pd.DataFrame(results)

# Correction for multiple comparisons
p_values = df_results['p_value'].values
reject, p_corrected, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

df_results['p_fdr'] = p_corrected
df_results['significant_fdr'] = reject

df_results = df_results.sort_values('p_fdr')

print(f"Number of significantly better products during the holiday season (FDR 5%): {df_results['significant_fdr'].sum()}\n")

print("Top 15 products with the biggest difference (statistically significant):")
print(df_results[df_results['significant_fdr']].head(15)[[
    'Description', 'mean_before', 'mean_after', 'diff', 'p_fdr', 'months_before', 'months_after'
]].round(2))